# Historical demonstration: execution disabled
This notebook includes random placeholder data, installation cells and local file writers. All former code cells are displayed as text for inspection. It is not a validated physics simulation. See the project README.


```python
import micropip
await micropip.install("numpy")
print("NumPy installed successfully!")
```


```python
%%writefile simulation.py
import numpy as np

class Attractor:
    def __init__(self, mass, center=None, area=None):
        self.mass = mass
        self.center = center
        self.area = area

class Simulator:
    def __init__(self, params):
        self.params = params
        self.attractor = None

    def run(self, t_total):
        self.attractor = Attractor(mass=np.random.uniform(0.1, 200))

    def detect_attractor(self):
        return self.attractor

    def apply_dynamic_fields(self, observer_dyn, rhythm_dyn):
        pass

    def detect_lag(self):
        return np.random.uniform(1, 100)

def run_pipeline(param_ranges, hyperopt_bounds=None):
    masses = np.logspace(-1, 2, num=10)
    lags = 50 * masses**(-0.08) * np.random.normal(1, 0.05, size=10)
    lag_errors = np.full_like(lags, 5.0)
    return masses, lags, lag_errors
```


```python
%%writefile visualization_module.py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import t, bootstrap
from piecewise_regression import Fit

# 1. Log–Log Regression with bootstrap CI
def plot_regression(masses, lags, n_boot=1000, ci=95):
    plt.figure(figsize=(8,6))
    sns.regplot(
        x=np.log10(masses),
        y=np.log10(lags),
        ci=ci,
        n_boot=n_boot,
        scatter_kws={'s': 50}
    )
    plt.xlabel('log10(mass)')
    plt.ylabel('log10(lag)')
    plt.title(f'Log–Log Regression with {ci}% Bootstrap CI')
    plt.show()

# 2. Residual diagnostics
def plot_residuals(masses, lags):
    def model(m, C, alpha): return C * m**alpha
    popt, _ = curve_fit(model, masses, lags, p0=[1.0, -0.1])
    residuals = lags - model(masses, *popt)
    plt.figure(figsize=(8,4))
    plt.scatter(masses, residuals, s=40)
    plt.axhline(0, color='grey', linestyle='--')
    plt.xscale('log')
    plt.xlabel('mass')
    plt.ylabel('Residual (lag – predicted)')
    plt.title('Residuals vs Mass')
    plt.show()

# 3. Parametric CI for α
def estimate_alpha_ci(masses, lags, lag_errors=None):
    def model(m, C, alpha): return C * m**alpha
    popt, pcov = curve_fit(
        model, masses, lags, p0=[1.0, -0.1],
        sigma=lag_errors, absolute_sigma=True if lag_errors is not None else False
    )
    alpha, alpha_err = popt[1], np.sqrt(np.diag(pcov))[1]
    dof = len(masses) - len(popt)
    tval = t.ppf(0.975, dof)
    ci_alpha = (alpha - alpha_err * tval, alpha + alpha_err * tval)
    return alpha, ci_alpha

# 4. Bootstrap CI for α
def bootstrap_alpha_ci(masses, lags, n_resamples=1000):
    def _stat(data):
        m, l = data
        popt, _ = curve_fit(lambda x, C, α: C * x**α, m, l, p0=[1.0, -0.1])
        return popt[1]
    res = bootstrap((masses, lags), _stat, confidence_level=0.95, n_resamples=n_resamples)
    return res.confidence_interval

# 5. Piecewise regression
def plot_piecewise(masses, lags, n_breakpoints=1, n_boot=100):
    x, y = np.log10(masses), np.log10(lags)
    pw = Fit(x, y, n_breakpoints=n_breakpoints, n_boot=n_boot)
    pw.plot_data(color="grey", s=20)
    pw.plot_fit(color="red", linewidth=2)
    pw.plot_breakpoints()
    pw.plot_breakpoint_confidence_intervals()
    plt.xlabel('log10(mass)')
    plt.ylabel('log10(lag)')
    plt.title('Segmented Regression with Breakpoint Detection')
    plt.show()
    return pw.get_results()
```


```python
import micropip

# Install all required libraries
await micropip.install([
    "matplotlib",
    "seaborn",
    "scipy",
    "piecewise-regression"
])

print("✅ Dependencies installed")
```


```python
import importlib
import simulation
import visualization_module as vm

importlib.reload(simulation)
importlib.reload(vm)

print("✅ Modules loaded successfully – we're ready to run simulations!")
```


```python
import numpy as np  # Ensure NumPy is available for this cell

# Step 7: Generate simulation data via your pipeline
param_ranges = {
    'lambda1': np.logspace(-2, 2, 5),
    'psi_amp': np.linspace(0.5, 2.0, 5),
    'psi_width': np.linspace(0.5, 2.0, 5),
}

# Run the simulation pipeline (stub version for now)
masses, lags, lag_errors = simulation.run_pipeline(param_ranges, hyperopt_bounds=None)

# Output results
print("✅ Simulation complete!")
print("Masses:", masses)
print("Lags:", lags)
print("Lag errors:", lag_errors)
```


```python
vm.plot_regression(masses, lags)
```


```python
vm.plot_residuals(masses, lags)
```


```python
alpha, param_ci = vm.estimate_alpha_ci(masses, lags, lag_errors=lag_errors)
print(f"Parametric α = {alpha:.3f}, 95% CI = {param_ci}")
```


```python
pw_results = vm.plot_piecewise(masses, lags)
print("Piecewise regression results:", pw_results)
```


```python
import numpy as np

# Generate 25 logarithmically-spaced mass targets between 1 and 3
log_min, log_max = np.log10(1.0), np.log10(3.0)  # ≈ 0.0 → 0.477
mass_targets = np.logspace(log_min, log_max, num=25)
print("Target masses:", mass_targets)
```


```python
%%writefile simulation.py
import numpy as np

# ... (keep your existing classes here) ...

def run_pipeline(param_ranges=None, mass_targets=None, tolerance=0.05):
    results = {'masses': [], 'lags': [], 'lag_errors': []}
    
    if mass_targets is None:
        # Fallback to original behavior
        mass_targets = np.logspace(-1,2,10)
    
    for target_m in mass_targets:
        # Quick stub: simulate attractor with mass near target (replace with real search logic)
        sim = Simulator(params={'lambda1':None})
        sim.run(t_total=100)
        m = sim.attractor.mass
        
        if abs(m - target_m) / target_m < tolerance:
            lag = sim.detect_lag()
            results['masses'].append(m)
            results['lags'].append(lag)
            results['lag_errors'].append(5.0)
    
    return np.array(results['masses']), np.array(results['lags']), np.array(results['lag_errors'])
```


```python
import importlib
import simulation
import numpy as np

importlib.reload(simulation)

# Use your mass_targets from earlier:
masses2, lags2, errors2 = simulation.run_pipeline(mass_targets=mass_targets)

print("Collected", len(masses2), "points")
print("Masses:", masses2)
print("Lags:", lags2)
```


```python
%%writefile simulation.py
import numpy as np

class Attractor:
    def __init__(self, mass, center=None, area=None):
        self.mass = mass; self.center = center; self.area = area

class Simulator:
    def __init__(self, params):
        self.params = params; self.attractor = None

    def run(self, t_total):
        self.attractor = Attractor(mass=np.random.uniform(1.0, 3.0))

    def detect_lag(self):
        return np.random.uniform(10, 60)

def run_pipeline(param_ranges=None, mass_targets=None, tolerance=0.2, attempts=10):
    results = {'masses': [], 'lags': [], 'lag_errors': []}
    if mass_targets is None:
        mass_targets = np.logspace(-1,2,10)

    for target in mass_targets:
        for _ in range(attempts):
            sim = Simulator({})
            sim.run(t_total=100)
            m = sim.attractor.mass
            if abs(m - target)/target < tolerance:
                results['masses'].append(m)
                results['lags'].append(sim.detect_lag())
                results['lag_errors'].append(5.0)
                break

    return (np.array(results['masses']),
            np.array(results['lags']),
            np.array(results['lag_errors']))
```


```python
import importlib, simulation, numpy as np
importlib.reload(simulation)

masses2, lags2, errors2 = simulation.run_pipeline(mass_targets=mass_targets)
print("Collected", len(masses2), "points")
print(masses2)
print(lags2)
```


```python
vm.plot_regression(masses2, lags2)
alpha2, ci2 = vm.estimate_alpha_ci(masses2, lags2, lag_errors=errors2)
print(f"Refined α = {alpha2:.3f}, 95% CI = {ci2}")
```


```python
pw2 = vm.plot_piecewise(masses2, lags2)
print("Refined piecewise results:", pw2)
```


```python

```
